##### 

##### Summary:
The notebook demonstrates LangChain prompting with Google Gemini and Groq models. It sets up the libraries and environment, then shows basic, zero-shot, one-shot, and few-shot prompting through arithmetic, translation, and emotion classification examples. It also introduces chain-of-thought and self-consistency prompting for reasoning tasks.

In [ ]:
# Install required libraries
# !pip install -U langchain
# !pip install langchain-google-genai
# !pip install -U langchain-groq

In [14]:
# Import necessary libraries
from dotenv import load_dotenv          # For loading environment variables from .env file
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnablePassthrough
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableSequence, RunnablePassthrough 

# Load environment variables from .env file
load_dotenv()           

True

##### Set up the LLM:
1. model_id: specifies which model you want to use
2. parameters: define the models configuration. If no custom parameters are passed to the function, the model will use default_params.
3. credentials: are required to successfully run LLMs
4. ChatGoogleGenerativeAI() is used to create an instance of the model

##### meaning of different parameters:
temparature: it controls the randomness and the creativity of the response. A lower temperature (e.g., 0.2) will make the output more focused and deterministic, while a higher temperature (e.g., 0.8) will make it more random and creative.


In [ ]:
# define a function to initialize and return a ChatGoogleGenerativeAI model
def llm_model(prompt_txt):
    """
    Initialize and return a ChatGoogleGenerativeAI model with specified parameters.
    
    Args:
        params (dict, optional): A dictionary containing model parameters such as 'temperature', 'max_output_tokens', 'top_p', and 'top_k'.
        
    Returns:
        ChatGoogleGenerativeAI: An instance of the ChatGoogleGenerativeAI model.
    """

    model_id = 'gemini-2.5-flash'

    # create a ChatGoogleGenerativeAI model with the specified parameters
    llm = ChatGoogleGenerativeAI(
        model = model_id,
        temperature = 0.7,        # Set the temperature for randomness in responses
        max_output_tokens = 1000, # Set the maximum number of tokens in the output
        top_p = 0.5,              # Set the top-p sampling parameter
        top_k = 1                 # Set the top-k sampling parameter
    )

    response = llm.invoke(prompt_txt)
    return response
   

In [3]:
# all other parameters are optional and can be set as needed and checked as below
# from google import genai
# import pprint

# client = genai.Client()

# # Query a specific model directly
# model_info = client.models.get(model='gemini-2.5-flash')

# # Pretty print the dictionary payload to see every parameter
# pprint.pprint(model_info)

##### Basic prompt:
 A basic prompt is the simplest form of prompting, where you provide a short text or phrase to the model without any special formatting or instructions. The model generates a continuation based on patterns it has learned during training. 

In [ ]:
# Make a call to the model
prompt = 'sum of 2 and 2 is?'

response = llm_model(prompt)

print("prompt is: ", prompt)
print("response is: ", response.text)

##### Zero Shot Prompt:
 No examples provided, just detailed instructions are needed to get the desired output. The model is expected to understand the instruction and generate the appropriate response based on its initial training and knowledge.


In [ ]:
# Make a call to the model using langchain_groq
llm = ChatGroq(model = "Qwen/Qwen3.8-27B")

prompt = """
        Classify the below sentence as True or False. If the sentence is True, return "True". If the sentence is False, return "False".
        Sentence: New Delhi is the capital of India.
"""

response = llm.invoke(prompt)
print(response.text)

##### One Shot Prompt:
you provide a single example and ask to do a similar task

In [ ]:
# make a call to the model using langchain_groq for translation from English to Hindi 
llm = ChatGroq(
    model = "Qwen/Qwen3.8-27B",
    temperature = 0,  
    max_tokens = None, 
    timeout = None,
    max_retries = 2
)

prompt = """
Here is an example of translating a sentence from English to Hindi:

English: I love holiday season.
Hindi: मुझे छुट्टियों का मौसम पसंद है।

Now translate the following sentence from English to Hindi:
English: I love to eat Indian breakfast.
"""

# Make a call to the model
response = llm.invoke(prompt)
print(response.text)

##### Few-Shot Prompt:
Few examples are provided to the model to understand the task and generate the appropriate response.

In [ ]:
# make a call to the model using langchain_groq for sentiment analysis of a sentence
llm = ChatGroq(
    model = "Qwen/Qwen3.8-27B",
    temperature = 0,  
    max_tokens = 5, 
    timeout = None,
    max_retries = 1
)

prompt = """Here are few examples of classifying emotions in statements:

            Statement: 'I am so happy to see you after so long.'
            Emotion: Joy
            
            Statement: 'oh no, I lost my sunglasses.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to a different country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement and return only the emotion in one word:
            Statement: 'I am so excited to start my new job tomorrow.'
            
"""

response = llm.invoke(prompt)
print(response.text)

Joy


##### Chain of thought:
The model is expected to reason step by step and provide a detailed explanation of its reasoning process before arriving at the final answer. This approach helps in understanding the model's thought process and can lead to more accurate and reliable responses. 

In [ ]:
# make a call to the model using langchain_groq for solving math and recipe problems(prompt as a list of prompts)
llm = ChatGroq(
    model = "Qwen/Qwen3.8-27B",
    temperature = 0.5,  
    max_tokens = 512, 
    timeout = None,
    max_retries = 1
)


# Lets create a list of prompts
prompt_math_question = """
                        Consider the problem: 'A sboy had 5 chocolates. He gave 2 to his friend. How many chocolates does he have left?'
                        
                        Break down each step of your calculation and don't use Latex formatting:

"""

prompt_noodle_recipe = """
                    Consider the problem: 'I want to make a noodle recipe. I have 2 cups of noodles, 1 cup of vegetables,
                    and 1/2 cup of sauce. I only want to use available ingredients and mix them. How many servings can I make?'
                    
                    Break down each step of your calculation and don't use Latex formatting:

"""

list_prompts = [prompt_math_question, prompt_noodle_recipe]

for prompt in list_prompts:
    response = llm.invoke(prompt)
    print("Prompt: ", prompt)
    print("Response: ", response.text)

##### Self-consistency:
An advanced technique that involves generating multiple responses to the same prompt and then selecting the most consistent answer among them. This can help improve the reliability of the model's output, especially for tasks that require reasoning or complex decision-making.



In [ ]:
# make a call to the model using langchain_groq for solving a math problem using self consistency prompting technique

llm = ChatGroq(
    model = "Qwen/Qwen3.8-27B",
    temperature = 0.5,  
    max_tokens = 512, 
    timeout = None,
    max_retries = 1
)

prompt = """
         when I was 4 years old, my sister was 8 year old. Now she is 46 years old. How old am I now?
         instructions: provide two independent calculations and explanations, then give me the most consistent answer.

"""

# Make a call to the model
response = llm.invoke(prompt)
print(f"prompt: {prompt}\n")
print(f"response: {response.text}")
